In [ ]:
# FrozenEstimator, which hands an already trained model to the calibrator, exists in scikit-learn from 1.6 on
%pip install -q "scikit-learn>=1.6"

# **Calibrating naive Bayes probabilities**

### A tutorial for the lesson “Naive Bayes models”

In the lesson “Naive Bayes models” we said two things: naive Bayes probabilities are pressed toward $0$ and $1$, so the number itself cannot be trusted, while the ordering of the classes and the direction of the feature contributions remain meaningful. Here we check both on data and see what calibration does to them.

Calibration is a monotone transformation of the model's score, trained on a separate split so that among the objects predicted at $p$ the share of class 1 is close to $p$. We go through two methods:

1. **Platt calibration** — a logistic function of the raw probability, $\hat p = \sigma(a\,p + b)$;
2. **isotonic calibration** — a non-decreasing step function fitted to the data.

And we check what calibration changes and what it does not: the probability number, the ordering of objects, the decision at a threshold and the feature contributions.

There are no self-check quizzes on this tutorial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42

## The data

We take the same Pima Indians Diabetes dataset as in the lesson “Random Forest: Practice”: eight numeric features, the target `Outcome` — whether the patient has diabetes. The calibrator must not be trained on the same rows as the model: on its training data the model's probabilities are biased more than on new data. So we split the sample into three parts.

In [ ]:
path = 'https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/diabetes.csv'
data = pd.read_csv(path)
X = data.drop(columns='Outcome')
y = data['Outcome']

# half to train the model, a quarter to train the calibrator, a quarter to test
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.5, random_state=RANDOM_STATE, stratify=y)
X_cal, X_test, y_cal, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest)
print('train:', len(X_train), ' calibration:', len(X_cal), ' test:', len(X_test))

## Naive Bayes and its contributions

Gaussian naive Bayes stores for every class $c$ and feature $i$ a mean $\mu_{c,i}$ and a variance $\sigma^2_{c,i}$. As in the lesson, the prediction decomposes into a sum on the logarithmic scale; for two classes it is convenient to write it as log-odds:

$$\log\frac{P(1\mid x)}{P(0\mid x)} = \log\frac{P(1)}{P(0)} + \sum_{i=1}^{n}\Big[\log P(x_i\mid 1) - \log P(x_i\mid 0)\Big].$$

The term in square brackets is the contribution of feature $i$. Let us check that the sigmoid of the sum of contributions gives `predict_proba` exactly.

In [ ]:
nb = GaussianNB().fit(X_train, y_train)

def contributions(model, X):
    """Feature contributions to the log-odds of class 1: log P(x_i | 1) - log P(x_i | 0) for Gaussian NB."""
    x = X.values[:, None, :]
    log_lik = -0.5 * np.log(2 * np.pi * model.var_)[None] - 0.5 * (x - model.theta_[None]) ** 2 / model.var_[None]
    return pd.DataFrame(log_lik[:, 1, :] - log_lik[:, 0, :], columns=X.columns, index=X.index)

prior = np.log(nb.class_prior_[1] / nb.class_prior_[0])
contrib = contributions(nb, X_test)
log_odds = prior + contrib.sum(axis=1)

p_raw = nb.predict_proba(X_test)[:, 1]
print('prior term log P(1)/P(0):', round(prior, 4))
print('largest gap between sigmoid(sum of contributions) and predict_proba:',
      np.abs(1 / (1 + np.exp(-log_odds)) - p_raw).max())

The gap is at the level of rounding error: the decomposition is exact, and it is what we read when we interpret naive Bayes.

## How far the probabilities can be trusted

A reliability curve compares the predicted probability with what actually happened. We split the test objects into five equally sized groups by predicted probability and in every group compare the mean predicted probability with the share of class 1. For a calibrated model the two columns coincide.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, p_raw, n_bins=5, strategy='quantile')
print(pd.DataFrame({'mean predicted probability': mean_pred.round(3),
                    'actual share of class 1': frac_pos.round(3)}))
print()
print('predictions below 0.01 or above 0.99:', int(((p_raw < 0.01) | (p_raw > 0.99)).sum()), 'of', len(p_raw))

In the lowest group the prediction and the share of class 1 almost coincide, but higher up they diverge: in the highest group the model promises more than $0.9$ on average, while class 1 makes up about two thirds of the objects there, and in the middle groups it underestimates instead. The probabilities are pushed to the edges further than the data justify — this is the pressing toward $0$ and $1$ from the lesson; some predictions go beyond $0.01$ and $0.99$.

## Platt and isotonic calibration

`CalibratedClassifierCV` trains a calibrator on top of a model. The `FrozenEstimator` wrapper tells it not to retrain naive Bayes but to take the trained one: this way the calibrator transforms the score of **one and the same** model, and the comparison before and after stays a comparison of one model. The calibrator is trained on the calibration part.

In [ ]:
platt = CalibratedClassifierCV(FrozenEstimator(nb), method='sigmoid').fit(X_cal, y_cal)
isotonic = CalibratedClassifierCV(FrozenEstimator(nb), method='isotonic').fit(X_cal, y_cal)

p_platt = platt.predict_proba(X_test)[:, 1]
p_iso = isotonic.predict_proba(X_test)[:, 1]

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='perfect calibration')
for name, p, color in [('naive Bayes, raw', p_raw, '#eb6834'),
                       ('Platt (sigmoid)', p_platt, '#2a78d6'),
                       ('isotonic', p_iso, '#1a9e5f')]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=5, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', color=color, label=name)
ax.set_xlabel('mean predicted probability of class 1')
ax.set_ylabel('actual share of class 1')
ax.set_title('Reliability curve on the test split, 5 quantile bins')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

Both calibrated curves are closer to the diagonal than the raw one overall, but not in every group: Platt calibration is a sigmoid with two parameters, and while lowering the upper groups it lifts the lowest one above its share of class 1. Let us compare with numbers: Brier and log-loss measure the quality of the probabilities themselves (lower is better), ROC-AUC measures only the ordering of objects.

In [ ]:
rows = {}
for name, p in [('no calibration', p_raw), ('Platt', p_platt), ('isotonic', p_iso)]:
    rows[name] = {
        'Brier': round(brier_score_loss(y_test, p), 4),
        'log-loss': round(log_loss(y_test, p), 4),
        'ROC-AUC': round(roc_auc_score(y_test, p), 4),
        'distinct values': len(np.unique(p)),
        'Spearman with raw': round(spearmanr(p_raw, p).statistic, 4),
        'changed class at 0.5': int(((p >= 0.5) != (p_raw >= 0.5)).sum()),
    }
pd.DataFrame(rows).T

What the table shows:

- **Platt** is a strictly increasing function of the raw probability. The ordering of objects does not change (the Spearman correlation with the raw probability is 1), so ROC-AUC is the same. The numbers themselves change: Brier and log-loss go down.
- **Isotonic** is a non-decreasing step function. There are as many distinct probability values as there are steps, objects within one step get the same probability and the ordering between them is lost — the Spearman correlation is below 1, ROC-AUC may move slightly. The extreme steps output probabilities $0$ or $1$, and log-loss, which penalizes a confident mistake hardest, can turn out worse than the raw one. These are the step artifacts of isotonic calibration, most visible on a small calibration split.
- Some objects **changed class** at the $0.5$ threshold. Does that mean calibration moved the decision boundary? Let us check.

In [ ]:
sig = platt.calibrated_classifiers_[0].calibrators[0]
t = -sig.b_ / sig.a_            # the raw probability that Platt maps exactly to 0.5
print('threshold 0.5 after Platt = threshold', round(t, 4), 'on the raw probability')
print('decisions agree on the whole test split:', bool(((p_platt >= 0.5) == (p_raw >= t)).all()))

The decisions agree: the $0.5$ threshold on the calibrated probability is a different threshold on the raw one. A monotone transformation creates no new decision boundary, it renames the thresholds: every boundary of the calibrated model is a boundary of the raw one at some threshold. The objects that changed class are the ones between the two thresholds.

## What calibration does to the feature contributions

The calibrator looks only at the final score, it does not see the features. So the contributions $\log P(x_i\mid 1) - \log P(x_i\mid 0)$ stay the same as in the raw model — both their ordering and their signs. But the sum of the contributions adds up to the raw probability, not to the calibrated one. Let us look at one test object.

In [ ]:
i = X_test.index[0]
row = contrib.loc[i]
print(row.sort_values(key=abs, ascending=False).round(3).to_string())
print()
print('prior term + sum of contributions (log-odds):', round(log_odds.loc[i], 4))
print('sigmoid(log-odds)                           :', round(1 / (1 + np.exp(-log_odds.loc[i])), 4))
print('raw probability predict_proba               :', round(p_raw[0], 4))
print('after Platt                                 :', round(p_platt[0], 4))
print('after isotonic                              :', round(p_iso[0], 4))
print()
gap = np.abs(1 / (1 + np.exp(-log_odds.values)) - p_platt)
print('on the whole test split: sigmoid(sum of contributions) differs from the Platt probability by',
      round(gap.mean(), 4), 'on average, the largest gap is', round(gap.max(), 4))

## What to take away

1. Platt and isotonic calibration are monotone transformations of the score on top of an already trained model. They fix the probability number, and the Brier score on the test split goes down.
2. Calibration preserves the ordering of objects (Platt strictly, isotonic up to the steps), and the decision boundary after calibration is a boundary of the raw model at a different threshold.
3. The additive decomposition on the probability scale breaks: the feature contributions explain the raw probability, and their sum no longer equals the calibrated prediction.
4. Calibration does not fix a wrong attribution: the feature contributions stay the contributions of the raw model, with all the consequences of the violated independence assumption. If the contributions could not be trusted before calibration, they cannot be trusted after.
5. Isotonic calibration produces step artifacts: identical probabilities for different objects and probabilities of $0$ and $1$ on the extreme steps.